# 01 - Data Preparation & Exploratory Data Analysis

## Objetivo
En este notebook se realiza:

- Extracción de datos desde la fuente oficial (MIDAGRI)
- Limpieza y transformación de la serie temporal
- Validación de la calidad de datos
- Análisis exploratorio de la serie
- Identificación de tendencia, estacionalidad y patrones
- Análisis de autocorrelación (ACF y PACF)

Este paso es fundamental para entender la estructura de la serie antes del modelado.

In [4]:
## Fuente
# Portal SIEA - MIDAGRI:
# https://siea.midagri.gob.pe/portal/publicacion/boletines-diarios/467-abastecimiento-y-precio-de-huevo

In [5]:
# Librerías base
import re
import io
import os
import time
import requests
import warnings
from pathlib import Path
from urllib.parse import urljoin, urlparse, parse_qs

# Manejo de datos
import numpy as np
import pandas as pd

# Parsing HTML
from bs4 import BeautifulSoup

# Lectura de PDF
import pdfplumber

# Visualización
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [6]:
SITE_URL = "https://siea.midagri.gob.pe"
BASE_URL = "https://siea.midagri.gob.pe/portal/publicacion/boletines-diarios/467-abastecimiento-y-precio-de-huevo"

YEAR_URLS = [
    "https://siea.midagri.gob.pe/portal/publicacion/boletines-diarios/467-abastecimiento-y-precio-de-huevo/468-2025",
    "https://siea.midagri.gob.pe/portal/publicacion/boletines-diarios/467-abastecimiento-y-precio-de-huevo/498-2026"
]

### Extraer meses

In [7]:
# Aquí guardaremos todos los meses
all_month_links = []

# Mozilla/5. es un User-Agent que ayuda a que la web no bloquee la solicitud, para q no nos salga error 403
headers = {"User-Agent": "Mozilla/5.0"}

# Recorremos cada año (2025 y 2026)
for year_url in YEAR_URLS:
        
    #  Pedimos el HTML de la página
    response = requests.get(year_url, headers=headers)
    
    #  Convertimos a formato BeautifulSoup
    soup = BeautifulSoup(response.text, "html.parser") #beautifulSoup hace q el formato html se pueda leer en python
    
    #  Sacamos todos los links <a href="...">
    links = []
    
    for a in soup.find_all("a"):
        if a.get("href") is not None:
            
            href = a.get("href")
            
            # Convertimos a link completo
            full_link = urljoin(SITE_URL, href)
            
            links.append(full_link)
    
    # Quitamos duplicados
    links = list(set(links))
    
    #  Filtramos solo los meses
    month_links = []
    
    for link in links:
        
        if BASE_URL in link:          # pertenece a la sección de huevo
            if link != year_url:      # no es la página del año
                if "download=" not in link:  # no es descarga
                    month_links.append(link)
    
    
    # Guardamos
    all_month_links.extend(month_links)

# Quitamos duplicados finales
all_month_links = list(set(all_month_links))

In [8]:
df_months = pd.DataFrame({"month_url": all_month_links})

df_months #hay 8 links

,month_url
0,https://siea.midagri.gob.pe/portal/publicacion...
1,https://siea.midagri.gob.pe/portal/publicacion...
2,https://siea.midagri.gob.pe/portal/publicacion...
3,https://siea.midagri.gob.pe/portal/publicacion...
4,https://siea.midagri.gob.pe/portal/publicacion...
5,https://siea.midagri.gob.pe/portal/publicacion...
6,https://siea.midagri.gob.pe/portal/publicacion...
7,https://siea.midagri.gob.pe/portal/publicacion...
8,https://siea.midagri.gob.pe/portal/publicacion...


In [9]:
# Aquí guardaremos todos los registros de todos los meses
all_daily_records = []

# Lista de nombres de meses para detectar fechas en texto
meses = [
    "enero", "febrero", "marzo", "abril", "mayo", "junio",
    "julio", "agosto", "setiembre", "septiembre",
    "octubre", "noviembre", "diciembre"
]

In [10]:
# recorrer mes a mes

for month_url in all_month_links:
    
    print(month_url)
    
    # 1. Descargar el HTML del mes
    response = requests.get(month_url, headers=headers)
    
    # 2. Convertir a BeautifulSoup
    soup = BeautifulSoup(response.text, "html.parser")
    
    # 3. Revisar todos los bloques div(organizan) de la página
    for div in soup.find_all("div"):
        
        texto = div.get_text(" ", strip=True) #extrae el texto dentro de cada div
        
        # 4. Nos quedamos solo con bloques que parecen boletines
        # porque contienen un mes en el texto y además la palabra DESCARGAR
        if any(mes in texto.lower() for mes in meses) and "descargar" in texto.lower():
            
            # 5. Buscar links dentro de ese bloque
            links_locales = []
            
            for a in div.find_all("a"):
                href = a.get("href")
                
                if href is not None:
                    link_completo = urljoin(SITE_URL, href)
                    links_locales.append(link_completo)
            
            # 6. Buscar el link de descarga del PDF
            download_url = None
            
            for link in links_locales:
                if "download=" in link:
                    download_url = link
                    break
            
            # 7. Guardar el registro
            all_daily_records.append({
                "month_url": month_url,
                "texto_boletin": texto,
                "download_url": download_url
            })

https://siea.midagri.gob.pe/portal/publicacion/boletines-diarios/467-abastecimiento-y-precio-de-huevo/468-2025/496-huevo-diciembre
https://siea.midagri.gob.pe/portal/publicacion/boletines-diarios/467-abastecimiento-y-precio-de-huevo
https://siea.midagri.gob.pe/portal/publicacion/boletines-diarios/467-abastecimiento-y-precio-de-huevo/468-2025/470-huevo-octubre
https://siea.midagri.gob.pe/portal/publicacion/boletines-diarios/467-abastecimiento-y-precio-de-huevo/468-2025/469-huevo-setiembre
https://siea.midagri.gob.pe/portal/publicacion/boletines-diarios/467-abastecimiento-y-precio-de-huevo/498-2026/499-huevo-enero
https://siea.midagri.gob.pe/portal/publicacion/boletines-diarios/467-abastecimiento-y-precio-de-huevo/498-2026/549-huevo-abril
https://siea.midagri.gob.pe/portal/publicacion/boletines-diarios/467-abastecimiento-y-precio-de-huevo/498-2026/536-huevo-marzo
https://siea.midagri.gob.pe/portal/publicacion/boletines-diarios/467-abastecimiento-y-precio-de-huevo/498-2026/524-huevo-febre

In [11]:
#convertimos a dataframe
df_daily_catalog = pd.DataFrame(all_daily_records)

In [12]:
df_daily_catalog

,month_url,texto_boletin,download_url
0,https://siea.midagri.gob.pe/portal/publicacion...,Inicio Nosotros Que es SIEA Instrumento de Ges...,https://siea.midagri.gob.pe/portal/publicacion...
1,https://siea.midagri.gob.pe/portal/publicacion...,Inicio Nosotros Que es SIEA Instrumento de Ges...,https://siea.midagri.gob.pe/portal/publicacion...
2,https://siea.midagri.gob.pe/portal/publicacion...,2025 Diciembre 2025 31 diciembre 2025 Hot DESC...,https://siea.midagri.gob.pe/portal/publicacion...
3,https://siea.midagri.gob.pe/portal/publicacion...,2025 Diciembre 2025 31 diciembre 2025 Hot DESC...,https://siea.midagri.gob.pe/portal/publicacion...
4,https://siea.midagri.gob.pe/portal/publicacion...,2025 Diciembre 2025 31 diciembre 2025 Hot DESC...,https://siea.midagri.gob.pe/portal/publicacion...
...,...,...,...
139,https://siea.midagri.gob.pe/portal/publicacion...,21 noviembre 2025 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...
140,https://siea.midagri.gob.pe/portal/publicacion...,20 noviembre 2025 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...
141,https://siea.midagri.gob.pe/portal/publicacion...,19 noviembre 2025 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...
142,https://siea.midagri.gob.pe/portal/publicacion...,18 noviembre 2025 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...


In [13]:
print("Total de registros encontrados:", len(df_daily_catalog))
print("Registros con link de descarga:", df_daily_catalog["download_url"].notna().sum())

Total de registros encontrados: 144
Registros con link de descarga: 144


In [14]:
#limpiamos duplicados
df_daily_catalog = df_daily_catalog.drop_duplicates().reset_index(drop=True)

print("Total después de quitar duplicados:", len(df_daily_catalog))

Total después de quitar duplicados: 104


In [15]:
df_daily_catalog

,month_url,texto_boletin,download_url
0,https://siea.midagri.gob.pe/portal/publicacion...,Inicio Nosotros Que es SIEA Instrumento de Ges...,https://siea.midagri.gob.pe/portal/publicacion...
1,https://siea.midagri.gob.pe/portal/publicacion...,2025 Diciembre 2025 31 diciembre 2025 Hot DESC...,https://siea.midagri.gob.pe/portal/publicacion...
2,https://siea.midagri.gob.pe/portal/publicacion...,2025 Diciembre 2025 31 diciembre 2025 Hot DESC...,https://siea.midagri.gob.pe/portal/publicacion...
3,https://siea.midagri.gob.pe/portal/publicacion...,31 diciembre 2025 Hot DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...
4,https://siea.midagri.gob.pe/portal/publicacion...,30 diciembre 2025 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...
...,...,...,...
99,https://siea.midagri.gob.pe/portal/publicacion...,21 noviembre 2025 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...
100,https://siea.midagri.gob.pe/portal/publicacion...,20 noviembre 2025 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...
101,https://siea.midagri.gob.pe/portal/publicacion...,19 noviembre 2025 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...
102,https://siea.midagri.gob.pe/portal/publicacion...,18 noviembre 2025 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...


Ya tenemos todos los pdfs, ahora normalizaremos los nombres

In [16]:
import re


In [17]:
fechas_extraidas = []

for texto in df_daily_catalog["texto_boletin"]:
    
    texto = str(texto).lower()
    
    patron = r"(\d{1,2}\s+(?:enero|febrero|marzo|abril|mayo|junio|julio|agosto|setiembre|septiembre|octubre|noviembre|diciembre)\s+\d{4})"
    
    match = re.search(patron, texto)
    
    if match:
        fechas_extraidas.append(match.group(1))
    else:
        fechas_extraidas.append(None)

df_daily_catalog["fecha_texto"] = fechas_extraidas

In [18]:
df_daily_catalog

,month_url,texto_boletin,download_url,fecha_texto
0,https://siea.midagri.gob.pe/portal/publicacion...,Inicio Nosotros Que es SIEA Instrumento de Ges...,https://siea.midagri.gob.pe/portal/publicacion...,25 diciembre 2025
1,https://siea.midagri.gob.pe/portal/publicacion...,2025 Diciembre 2025 31 diciembre 2025 Hot DESC...,https://siea.midagri.gob.pe/portal/publicacion...,25 diciembre 2025
2,https://siea.midagri.gob.pe/portal/publicacion...,2025 Diciembre 2025 31 diciembre 2025 Hot DESC...,https://siea.midagri.gob.pe/portal/publicacion...,25 diciembre 2025
3,https://siea.midagri.gob.pe/portal/publicacion...,31 diciembre 2025 Hot DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,31 diciembre 2025
4,https://siea.midagri.gob.pe/portal/publicacion...,30 diciembre 2025 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,30 diciembre 2025
...,...,...,...,...
99,https://siea.midagri.gob.pe/portal/publicacion...,21 noviembre 2025 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,21 noviembre 2025
100,https://siea.midagri.gob.pe/portal/publicacion...,20 noviembre 2025 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,20 noviembre 2025
101,https://siea.midagri.gob.pe/portal/publicacion...,19 noviembre 2025 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,19 noviembre 2025
102,https://siea.midagri.gob.pe/portal/publicacion...,18 noviembre 2025 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,18 noviembre 2025


In [19]:
#eliminamos filas sin fecha o sin pdf
df_daily_catalog = df_daily_catalog[
    df_daily_catalog["fecha_texto"].notna() &
    df_daily_catalog["download_url"].notna()
].reset_index(drop=True)

print("Total final de boletines válidos:", len(df_daily_catalog))

Total final de boletines válidos: 104


In [20]:
df_daily_catalog

,month_url,texto_boletin,download_url,fecha_texto
0,https://siea.midagri.gob.pe/portal/publicacion...,Inicio Nosotros Que es SIEA Instrumento de Ges...,https://siea.midagri.gob.pe/portal/publicacion...,25 diciembre 2025
1,https://siea.midagri.gob.pe/portal/publicacion...,2025 Diciembre 2025 31 diciembre 2025 Hot DESC...,https://siea.midagri.gob.pe/portal/publicacion...,25 diciembre 2025
2,https://siea.midagri.gob.pe/portal/publicacion...,2025 Diciembre 2025 31 diciembre 2025 Hot DESC...,https://siea.midagri.gob.pe/portal/publicacion...,25 diciembre 2025
3,https://siea.midagri.gob.pe/portal/publicacion...,31 diciembre 2025 Hot DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,31 diciembre 2025
4,https://siea.midagri.gob.pe/portal/publicacion...,30 diciembre 2025 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,30 diciembre 2025
...,...,...,...,...
99,https://siea.midagri.gob.pe/portal/publicacion...,21 noviembre 2025 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,21 noviembre 2025
100,https://siea.midagri.gob.pe/portal/publicacion...,20 noviembre 2025 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,20 noviembre 2025
101,https://siea.midagri.gob.pe/portal/publicacion...,19 noviembre 2025 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,19 noviembre 2025
102,https://siea.midagri.gob.pe/portal/publicacion...,18 noviembre 2025 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,18 noviembre 2025


DESARGAR LOS PDFS

In [22]:
import os
os.getcwd()
os.listdir()

['01_leer_y_preprocesar_data.ipynb']

In [23]:
df_daily_catalog

,month_url,texto_boletin,download_url,fecha_texto
0,https://siea.midagri.gob.pe/portal/publicacion...,Inicio Nosotros Que es SIEA Instrumento de Ges...,https://siea.midagri.gob.pe/portal/publicacion...,25 diciembre 2025
1,https://siea.midagri.gob.pe/portal/publicacion...,2025 Diciembre 2025 31 diciembre 2025 Hot DESC...,https://siea.midagri.gob.pe/portal/publicacion...,25 diciembre 2025
2,https://siea.midagri.gob.pe/portal/publicacion...,2025 Diciembre 2025 31 diciembre 2025 Hot DESC...,https://siea.midagri.gob.pe/portal/publicacion...,25 diciembre 2025
3,https://siea.midagri.gob.pe/portal/publicacion...,31 diciembre 2025 Hot DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,31 diciembre 2025
4,https://siea.midagri.gob.pe/portal/publicacion...,30 diciembre 2025 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,30 diciembre 2025
...,...,...,...,...
99,https://siea.midagri.gob.pe/portal/publicacion...,21 noviembre 2025 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,21 noviembre 2025
100,https://siea.midagri.gob.pe/portal/publicacion...,20 noviembre 2025 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,20 noviembre 2025
101,https://siea.midagri.gob.pe/portal/publicacion...,19 noviembre 2025 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,19 noviembre 2025
102,https://siea.midagri.gob.pe/portal/publicacion...,18 noviembre 2025 DESCARGAR VISUALIZAR,https://siea.midagri.gob.pe/portal/publicacion...,18 noviembre 2025


In [24]:
import os
pdf_folder = "../pdfs"

DESCARGAMOS solo lo nuevo para evitar descargar todos los pdfs cada que abrimos el código

In [27]:
import requests
import os

for i, row in df_daily_catalog.iterrows():
    url = row["download_url"]
    fecha = row["fecha_texto"]
    
    fecha_clean = fecha.replace(" ", "_").replace("/", "-")
    file_path = f"{pdf_folder}/{fecha_clean}.pdf"
    
    if os.path.exists(file_path):
        print(f"Saltando {fecha} (ya existe)")
        continue
    
    print(f"Descargando {fecha}")
    try:
        response = requests.get(url)
        with open(file_path, "wb") as f:
            f.write(response.content)
        print(f" Descargado")
    except Exception as e:
        print(f"Error: {e}")

Saltando 25 diciembre 2025 (ya existe)
Saltando 25 diciembre 2025 (ya existe)
Saltando 25 diciembre 2025 (ya existe)
Saltando 31 diciembre 2025 (ya existe)
Saltando 30 diciembre 2025 (ya existe)
Saltando 29 diciembre 2025 (ya existe)
Saltando 22 diciembre 2025 (ya existe)
Saltando 19 diciembre 2025 (ya existe)
Saltando 18 diciembre 2025 (ya existe)
Saltando 17 diciembre 2025 (ya existe)
Saltando 16 diciembre 2025 (ya existe)
Saltando 15 diciembre 2025 (ya existe)
Saltando 12 diciembre 2025 (ya existe)
Saltando 25 octubre 2025 (ya existe)
Saltando 25 octubre 2025 (ya existe)
Saltando 25 octubre 2025 (ya existe)
Saltando 31 octubre 2025 (ya existe)
Saltando 30 octubre 2025 (ya existe)
Saltando 29 octubre 2025 (ya existe)
Saltando 28 octubre 2025 (ya existe)
Saltando 27 octubre 2025 (ya existe)
Saltando 24 octubre 2025 (ya existe)
Saltando 23 octubre 2025 (ya existe)
Saltando 22 octubre 2025 (ya existe)
Saltando 21 octubre 2025 (ya existe)
Saltando 20 octubre 2025 (ya existe)
Saltando 25 

lista de los archivos pdfs

In [29]:
pdf_folder = "../pdfs"

# Lista de archivos PDF
pdf_files = [f for f in os.listdir(pdf_folder) if f.lower().endswith(".pdf")]
pdf_files

['09_abril_2026.pdf',
 '10_abril_2026.pdf',
 '12_diciembre_2025.pdf',
 '13_abril_2026.pdf',
 '13_febrero_2026.pdf',
 '14_abril_2026.pdf',
 '15_abril_2026.pdf',
 '15_diciembre_2025.pdf',
 '16_abril_2026.pdf',
 '16_diciembre_2025.pdf',
 '16_enero_2026.pdf',
 '16_setiembre_2025.pdf',
 '17_abril_2026.pdf',
 '17_diciembre_2025.pdf',
 '17_febrero_2026.pdf',
 '17_noviembre_2025.pdf',
 '17_setiembre_2025.pdf',
 '18_diciembre_2025.pdf',
 '18_febrero_2026.pdf',
 '18_marzo_2026.pdf',
 '18_noviembre_2025.pdf',
 '18_setiembre_2025.pdf',
 '19_diciembre_2025.pdf',
 '19_enero_2026.pdf',
 '19_febrero_2026.pdf',
 '19_marzo_2026.pdf',
 '19_noviembre_2025.pdf',
 '19_setiembre_2025.pdf',
 '20_abril_2026.pdf',
 '20_enero_2026.pdf',
 '20_febrero_2026.pdf',
 '20_marzo_2026.pdf',
 '20_noviembre_2025.pdf',
 '20_octubre_2025.pdf',
 '21_abril_2026.pdf',
 '21_enero_2026.pdf',
 '21_noviembre_2025.pdf',
 '21_octubre_2025.pdf',
 '22_abril_2026.pdf',
 '22_diciembre_2025.pdf',
 '22_enero_2026.pdf',
 '22_octubre_2025.pd

# LECTURA DE LAS 3 TABLAS

## PRECIOS DE HUEVO SEGUN TIPO DE COMERCIALIZACION (Soles por kilogramo)


In [35]:
import re
import pandas as pd
import pdfplumber
import os
from tqdm import tqdm

# Mapeo de meses
meses_map = {'ene':1, 'feb':2, 'mar':3, 'abr':4, 'may':5, 'jun':6,
             'jul':7, 'ago':8, 'sep':9, 'oct':10, 'nov':11, 'dic':12}

# Archivo para guardar qué PDFs ya procesamos
PROCESADOS_FILE = "pdfs_procesados.txt"

# Cargar lista de PDFs ya procesados
if os.path.exists(PROCESADOS_FILE):
    with open(PROCESADOS_FILE, "r") as f:
        procesados = set(line.strip() for line in f)
else:
    procesados = set()

# Lista para guardar datos
registros_tabla1 = []

print(f"PDFs ya procesados: {len(procesados)}")

PDFs ya procesados: 0


solo procesaremos a texto lospdfs nuevos ;)

In [36]:
# Filtrar PDFs que NO han sido procesados
pdfs_nuevos = [pdf for pdf in pdf_files if pdf not in procesados]

print(f" Total PDFs: {len(pdf_files)}")
print(f" PDFs nuevos por procesar: {len(pdfs_nuevos)}")

# Procesar solo los nuevos
for pdf_name in tqdm(pdfs_nuevos, desc="Procesando PDFs nuevos"):
    pdf_path = os.path.join(pdf_folder, pdf_name)
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            text = pdf.pages[0].extract_text()
        
        lines = text.split("\n")
        patron = r"^(ene|feb|mar|abr|may|jun|jul|ago|sep|oct|nov|dic)\s+(\d{1,2})\s+(\d+\.\d+)\s+(\d+\.\d+)$"
        
        for line in lines:
            line = line.strip().lower()
            match = re.match(patron, line)
            if match:
                registros_tabla1.append({
                    "pdf_name": pdf_name,
                    "mes": match.group(1),
                    "mes_num": meses_map[match.group(1)],
                    "dia": int(match.group(2)),
                    "mayorista": float(match.group(3)),
                    "minorista": float(match.group(4))
                })
        
        # Marcar este PDF como procesado
        procesados.add(pdf_name)
        with open(PROCESADOS_FILE, "a") as f:
            f.write(pdf_name + "\n")
            
    except Exception as e:
        print(f"Error en {pdf_name}: {e}")

print(f" Nuevos registros extraídos: {len(registros_tabla1)}")

 Total PDFs: 83
 PDFs nuevos por procesar: 83


Procesando PDFs nuevos: 100%|██████████| 83/83 [05:36<00:00,  4.05s/it]

 Nuevos registros extraídos: 1650


In [37]:
#creamos el datagrame
df_tabla1 = pd.DataFrame(registros_tabla1)
df_tabla1.head(20)

,pdf_name,mes,mes_num,dia,mayorista,minorista
0,09_abril_2026.pdf,mar,3,27,6.25,7.53
1,09_abril_2026.pdf,mar,3,28,6.25,7.53
2,09_abril_2026.pdf,mar,3,29,6.25,7.53
3,09_abril_2026.pdf,mar,3,30,6.18,7.45
4,09_abril_2026.pdf,mar,3,31,6.05,7.45
5,09_abril_2026.pdf,abr,4,1,5.95,7.31
6,09_abril_2026.pdf,abr,4,2,5.95,7.31
7,09_abril_2026.pdf,abr,4,3,5.95,7.31
8,09_abril_2026.pdf,abr,4,4,5.95,7.31
9,09_abril_2026.pdf,abr,4,5,5.95,7.31


In [40]:
# df_tabla1 debería ser guardado en csv para evitar correr todo el proceso
df_tabla1.to_csv("datos_tabla1.csv", index=False)

## OFERTA DEL HUEVO DE PRIMERA SEGUN MACROREGION(toneladas)


In [38]:
# Mapeo de meses
meses_map = {'ene':1, 'feb':2, 'mar':3, 'abr':4, 'may':5, 'jun':6,
             'jul':7, 'ago':8, 'sep':9, 'oct':10, 'nov':11, 'dic':12}

# Archivo para controlar PDFs ya procesados en Tabla 2
PROCESADOS_FILE = "pdfs_procesados_tabla2.txt"

# Cargar PDFs ya procesados
if os.path.exists(PROCESADOS_FILE):
    with open(PROCESADOS_FILE, "r") as f:
        procesados = set(line.strip() for line in f)
else:
    procesados = set()

registros_tabla2 = []

print(f"PDFs ya procesados: {len(procesados)}")

PDFs ya procesados: 0


Preprocesar solo pdfs nuevos

In [39]:
# Filtrar PDFs nuevos
pdfs_nuevos = [pdf for pdf in pdf_files if pdf not in procesados]

print(f" Total PDFs: {len(pdf_files)}")
print(f" PDFs nuevos: {len(pdfs_nuevos)}")

for pdf_name in tqdm(pdfs_nuevos, desc="Procesando Tabla 2"):
    pdf_path = os.path.join(pdf_folder, pdf_name)
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            text = pdf.pages[0].extract_text()
        
        lines = text.split("\n")
        
        for line in lines:
            line_lower = line.strip().lower()
            
            # Patrón: mes + día + números
            patron_inicio = r"^(ene|feb|mar|abr|may|jun|jul|ago|sep|oct|nov|dic)\s+(\d{1,2})\s+"
            match = re.match(patron_inicio, line_lower)
            
            if match:
                mes = match.group(1)
                dia = int(match.group(2))
                
                # Extraer números
                resto = re.sub(patron_inicio, "", line_lower)
                partes = resto.split()
                
                valores = []
                for x in partes:
                    x = x.replace(",", "")
                    if x in ["-", "#n/a"]:
                        valores.append(np.nan)
                    else:
                        try:
                            valores.append(float(x))
                        except:
                            pass
                
                # Guardar primeras 6 columnas
                if len(valores) >= 6:
                    registros_tabla2.append({
                        "pdf_name": pdf_name,
                        "mes": mes,
                        "mes_num": meses_map[mes],
                        "dia": dia,
                        "oferta_lima": valores[0],
                        "oferta_sierra": valores[1],
                        "oferta_costa_norte": valores[2],
                        "oferta_selva": valores[3],
                        "oferta_costa_sur": valores[4],
                        "oferta_total": valores[5]
                    })
        
        # Marcar como procesado
        procesados.add(pdf_name)
        with open(PROCESADOS_FILE, "a") as f:
            f.write(pdf_name + "\n")
            
    except Exception as e:
        print(f" Error en {pdf_name}: {e}")

print(f" Nuevos registros: {len(registros_tabla2)}")

 Total PDFs: 83
 PDFs nuevos: 83


Procesando Tabla 2: 100%|██████████| 83/83 [05:06<00:00,  3.70s/it]

 Nuevos registros: 1134


Guardar resultados

In [41]:
df_tabla2 = pd.DataFrame(registros_tabla2)

# Cargar datos anteriores si existen
if os.path.exists("datos_tabla2.csv"):
    df_anterior = pd.read_csv("datos_tabla2.csv")
    df_tabla2 = pd.concat([df_anterior, df_tabla2], ignore_index=True)
    print(f" Datos anteriores: {len(df_anterior)} registros")

print(f" Total acumulado: {len(df_tabla2)} registros")



 Total acumulado: 1134 registros


In [42]:
# Guardar
df_tabla2.to_csv("datos_tabla2.csv", index=False)


## PRECIO DEL HUEVO DE PRIMERA CALIDAD EN CENTRO DE PRODUCCION (Soles por Kg)


In [43]:


# Mapeo de meses
meses_map = {'ene':1, 'feb':2, 'mar':3, 'abr':4, 'may':5, 'jun':6,
             'jul':7, 'ago':8, 'sep':9, 'oct':10, 'nov':11, 'dic':12}

# Archivo para controlar PDFs ya procesados en Tabla 3
PROCESADOS_FILE = "pdfs_procesados_tabla3.txt"

# Cargar PDFs ya procesados
if os.path.exists(PROCESADOS_FILE):
    with open(PROCESADOS_FILE, "r") as f:
        procesados = set(line.strip() for line in f)
else:
    procesados = set()

registros_tabla3 = []

print(f" PDFs ya procesados (Tabla 3): {len(procesados)}")

 PDFs ya procesados (Tabla 3): 0


Procesamos solo los pdfs nuevos

In [44]:
# Filtrar PDFs nuevos
pdfs_nuevos = [pdf for pdf in pdf_files if pdf not in procesados]

print(f" Total PDFs: {len(pdf_files)}")
print(f" PDFs nuevos: {len(pdfs_nuevos)}")

for pdf_name in tqdm(pdfs_nuevos, desc="Procesando Tabla 3"):
    pdf_path = os.path.join(pdf_folder, pdf_name)
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            text = pdf.pages[0].extract_text()
        
        lines = text.split("\n")
        
        for line in lines:
            line_lower = line.strip().lower()
            
            # Patrón: mes + día + números
            patron_inicio = r"^(ene|feb|mar|abr|may|jun|jul|ago|sep|oct|nov|dic)\s+(\d{1,2})\s+"
            match = re.match(patron_inicio, line_lower)
            
            if match:
                mes = match.group(1)
                dia = int(match.group(2))
                
                # Extraer números
                resto = re.sub(patron_inicio, "", line_lower)
                partes = resto.split()
                
                valores = []
                for x in partes:
                    x = x.replace(",", "")
                    if x in ["-", "#n/a"]:
                        valores.append(np.nan)
                    else:
                        try:
                            valores.append(float(x))
                        except:
                            pass
                
                # Asignar según cantidad de valores (Ica, La Libertad, Lima)
                if len(valores) >= 3:
                    precio_ica = valores[0]
                    precio_la_libertad = valores[1]
                    precio_lima = valores[2]
                elif len(valores) == 2:
                    precio_ica = valores[0]
                    precio_la_libertad = np.nan
                    precio_lima = valores[1]
                elif len(valores) == 1:
                    precio_ica = valores[0]
                    precio_la_libertad = np.nan
                    precio_lima = np.nan
                else:
                    continue
                
                registros_tabla3.append({
                    "pdf_name": pdf_name,
                    "mes": mes,
                    "mes_num": meses_map[mes],
                    "dia": dia,
                    "precio_ica": precio_ica,
                    "precio_la_libertad": precio_la_libertad,
                    "precio_lima": precio_lima
                })
        
        # Marcar como procesado
        procesados.add(pdf_name)
        with open(PROCESADOS_FILE, "a") as f:
            f.write(pdf_name + "\n")
            
    except Exception as e:
        print(f" Error en {pdf_name}: {e}")

print(f" Nuevos registros Tabla 3: {len(registros_tabla3)}")

 Total PDFs: 83
 PDFs nuevos: 83


Procesando Tabla 3: 100%|██████████| 83/83 [03:13<00:00,  2.33s/it]

 Nuevos registros Tabla 3: 3566


guardamos en csv

In [45]:
df_tabla3 = pd.DataFrame(registros_tabla3)

# Cargar datos anteriores si existen
if os.path.exists("datos_tabla3.csv"):
    df_anterior = pd.read_csv("datos_tabla3.csv")
    df_tabla3 = pd.concat([df_anterior, df_tabla3], ignore_index=True)
    print(f"📚 Datos anteriores: {len(df_anterior)} registros")

print(f" Total acumulado Tabla 3: {len(df_tabla3)} registros")




 Total acumulado Tabla 3: 3566 registros


In [46]:
# Guardar
df_tabla3.to_csv("datos_tabla3.csv", index=False)